# Gross Margin Tracking for Top U.S. Semiconductor Companies

**NOTE**: Edit HEADERS in the first code cell with your name and email (required by the SEC).

## Company names and trading symbols

| Name | Symbol |
|:------------|:-----:|
| NVIDIA Corporation | NVDA |
| Broadcom Inc. | AVGO |
| Intel Corporation | INTC |
| QUALCOMM Incorporated | QCOM |
| Advanced Micro Devices, Inc. | AMD |
| Micron Technology, Inc. | MU |
| Applied Materials, Inc. | AMAT |
| Lam Research Corporation | LRCX |
| KLA Corporation | KLAC |

Sources:
  - NVIDIA, logo.dev (https://www.logo.dev/search/brands/nvidia.com)
  - AMD, Brandpalettes (https://brandpalettes.com/amd-colors/)
  - Broadcom, Designpieces
    (https://www.designpieces.com/palette/broadcom-color-palette-hex-and-rgb/)
  - Intel, Designpieces
    (https://www.designpieces.com/palette/intel-color-palette-hex-and-rgb/)
  - Micron, logo.dev (https://www.logo.dev/search/brands/micron.com)
  - Applied Materials, logo.dev (https://www.logo.dev/search/brands/appliedmaterials.com)
  - Lam Research, Brandpalettes (https://brandpalettes.com/lam-research-colors/)
  - KLA, Brand Color Code (https://www.brandcolorcode.com/kla-corporation)
  - Qualcomm (https://www.designpieces.com/palette/qualcomm-color-palette-hex-and-rgb/)


In [ ]:
import time
import requests
import pandas as pd
import polars as pl

import great_tables as gt

from plotnine import *
from mizani.labels import label_percent

# SEC requires a descriptive User-Agent with contact info on every request
HEADERS = {"User-Agent": "Your Name (name@email.com)"}

tickers = ["NVDA", "AVGO", "INTC", "QCOM", "AMD", "MU", "AMAT", "LRCX", "KLAC"]

# Map ticker -> CIK (SEC's company identifier)
resp = requests.get("https://www.sec.gov/files/company_tickers.json", headers=HEADERS)
resp.raise_for_status()
lookup = pd.DataFrame(resp.json()).T
ciks = lookup[lookup["ticker"].isin(tickers)].set_index("ticker")["cik_str"].astype(int)
ciks

In [ ]:
def get_company_facts(cik):
    """Download every XBRL fact a company has filed (one JSON per company)."""
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json"
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    time.sleep(0.2)  # stay well under SEC's 10 requests/second limit
    return r.json()

facts = {ticker: get_company_facts(cik) for ticker, cik in ciks.items()}

In [ ]:
# Companies tag the same line item differently, so try tags in priority order
REVENUE_TAGS = ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet"]
GROSS_PROFIT_TAGS = ["GrossProfit"]
COST_TAGS = ["CostOfRevenue", "CostOfGoodsAndServicesSold"]


def annual_values(company_facts, tags):
    """Full-year values from 10-Ks, indexed by fiscal year-end date."""
    rows = []
    for rank, tag in enumerate(tags):
        for r in company_facts["facts"].get("us-gaap", {}).get(tag, {}).get("units", {}).get("USD", []):
            if r["form"] != "10-K" or "start" not in r:
                continue
            days = (pd.Timestamp(r["end"]) - pd.Timestamp(r["start"])).days
            if 350 <= days <= 380:  # keep 12-month periods (52/53-week years included)
                rows.append({"end": pd.Timestamp(r["end"]), "val": r["val"],
                             "filed": pd.Timestamp(r["filed"]), "rank": rank})
    if not rows:
        return pd.Series(dtype=float)
    df = pd.DataFrame(rows).sort_values(["end", "rank", "filed"], ascending=[True, True, False])
    return df.drop_duplicates("end").set_index("end")["val"]  # preferred tag, latest filing


frames = []
for ticker, f in facts.items():
    revenue = annual_values(f, REVENUE_TAGS)
    gross_profit = annual_values(f, GROSS_PROFIT_TAGS)
    cost = annual_values(f, COST_TAGS)
    # Some filers (e.g. Qualcomm) don't report GrossProfit, so fall back to revenue - cost
    gross_profit = gross_profit.combine_first(revenue - cost)
    out = pd.DataFrame({"revenue": revenue, "gross_profit": gross_profit}).dropna()
    out["gross_margin"] = out["gross_profit"] / out["revenue"]
    out.index.name = "fiscal_year_end"
    frames.append(out.reset_index().assign(ticker=ticker))

gross_margins = pd.concat(frames, ignore_index=True)[["ticker", "fiscal_year_end", "revenue", "gross_profit", "gross_margin"]]
gross_margins.groupby("ticker").tail(3)

In [ ]:
# Lookup table kept separate from the EDGAR pull; merged in for plotting
sectors = pd.DataFrame({
    "ticker":  ["NVDA", "AMD", "AVGO", "QCOM", "INTC", "MU", "AMAT", "LRCX", "KLAC"],
    "company": ["NVIDIA", "AMD", "Broadcom", "Qualcomm", "Intel", "Micron", "Applied Materials", "Lam Research", "KLA"],
    "sector":  ["Fabless", "Fabless", "Fabless", "Fabless", "IDM", "IDM", "WFE", "WFE", "WFE"],
})
# Ordered category so facets/legends follow this order instead of alphabetical
sectors["sector"] = pd.Categorical(sectors["sector"], categories=["Fabless", "IDM", "WFE"], ordered=True)

# validate="m:1" raises if a ticker is duplicated in the lookup; the check catches missing ones
plot_df = gross_margins.merge(sectors, on="ticker", how="left", validate="m:1")
assert plot_df["sector"].notna().all(), "ticker missing from sectors table"
plot_df.groupby(["sector", "company"], observed=True).size()


In [ ]:
# Keys match the "company" column. Where brands share a hue within a panel, one is
# swapped for a distinguishable color so the lines stay readable.
brand_colors = {
    "NVIDIA":            "#76B900",  # NVIDIA green
    "AMD":               "#000000",  # AMD's logo is black (its red would clash with Broadcom)
    "Broadcom":          "#CC092F",  # Broadcom red
    "Qualcomm":          "#2a2aea",  # Qualcomm blue
    "Intel":             "#0068B5",  # Intel blue
    "Micron":            "#6E0890",  # violet; Micron's blue would clash with Intel
    "Applied Materials": "#347DA2",  # steel blue
    "Lam Research":      "#000000",  # Lam logo is black
    "KLA":               "#41007F",  # KLA purple
}

MIN_GAP = 0.04  # smallest vertical distance between labels, in gross-margin units


def spread(y, gap):
    """Push each value up so it sits at least gap above the one before it."""
    out = []
    for v in y:
        out.append(v if not out else max(v, out[-1] + gap))
    return out


# One row per company (its most recent fiscal year) with a de-overlapped label height
labels = (
    plot_df.sort_values("fiscal_year_end").groupby("ticker").tail(1)
    .sort_values(["sector", "gross_margin"])
)
labels["label_y"] = labels.groupby("sector", observed=True)["gross_margin"].transform(
    lambda s: pd.Series(spread(s, MIN_GAP), index=s.index)
)

(
    plot_df
    >>
    ggplot(aes("fiscal_year_end", "gross_margin", color="company"))
    + geom_line(size=1)
    + geom_text(
        aes(y="label_y", label="ticker"),
        data=labels,
        ha="left",
        nudge_x=120,  # days: moves the label just right of the line end
        size=10
    )
    + labs(
        subtitle = "Gross Margin",
        caption = (
            "Source: https://data.sec.gov\n"
            "IDM = Integrated Device Manufacturer\n"
            "WFE = Wafer Fab Equipment supplier"
        )
    )
    + scale_x_date(
        date_labels="%Y",
        expand=(0.02, 0, 0.2, 0)  # extra room on the right so labels aren't clipped
    )
    + scale_y_continuous(labels=label_percent())
    + scale_color_manual(values=brand_colors)
    + facet_wrap("sector", nrow=1)
    + theme_light()
    + theme(
        figure_size=(8, 4.5),
        legend_position="none",  # labels replace the legend
        axis_title=element_blank(),
        axis_text=element_text(size=10),
        plot_title_position="plot",
        plot_caption=element_text(linespacing=1.1, ha="left"),
        plot_caption_position="plot",
        strip_background=element_rect(fill="#117ac9"),
        strip_text=element_text(size=10)
    )
)

In [ ]:
(
    gt.GT(
        pl.DataFrame(gross_margins)
            .group_by(pl.col('ticker'))
            .last()
        .sort(pl.col('gross_margin'), descending=True)
    ).tab_header(
        title = "Gross margins at leading U.S. semiconductor companies",
        subtitle = "Latest fiscal year, sorted by gross margin"
    ).fmt_percent(columns='gross_margin', decimals=1)
    .fmt_currency(columns=['revenue', 'gross_profit'], scale_by=1e-9, decimals=0)
    .fmt_date(columns="fiscal_year_end", date_style="m_day_year")
    .cols_label(
        ticker='Ticker',
        fiscal_year_end="Fiscal year end",
        revenue="Revenue (Billions)",
        gross_profit='Gross profit (Billions)',
        gross_margin='Gross margin (%)'
    )
)